# A FAIR² Mental Health Survey Dataset from Kilifi County, Kenya Exploration with `mlcroissant`
This notebook guides you through loading, exploring, and processing the FAIR² Mental Health Survey Dataset from Kilifi County, Kenya using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` and visualization libraries are installed
!pip install mlcroissant[full] matplotlib seaborn --quiet

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}")

### Dataset details
* Identifier: 10.71728/senscience.vcs2-05nj
* Version: 1.0
* License: https://opendatacommons.org/licenses/by/1-0/
* Collection timeframe: 2024-08-15 to 2024-09-10
* Main Keywords: demographic data, GAD-7, Kenya, Kilifi County, mental health, PHQ-9, PSQ
* Data summary: Psychological and demographic survey data (including GAD-7, PHQ-9, PSQ scores)

You can read more about the dataset and its documentation at the [FAIR² data page](https://sen.science/doi/10.71728/senscience.vcs2-05nj).

## 2. Data Overview
Review available record sets, their `@id`s, and fields.

In [ ]:
# List all record sets by @id and fields

record_set_ids = list(dataset.record_sets.keys())
print(f"Record sets (@id):\n-------------------")
for rs_id in record_set_ids:
    rs = dataset.record_sets[rs_id]
    print(f"RecordSet: {rs_id}")
    if hasattr(rs, 'fields'):
        if rs.fields:
            print("  Fields:")
            for field in rs.fields:
                print(f"    - {field['@id']} (name: {field['name']})")
        else:
            print("  No fields found in this record set.")
    print()
# For demonstration, print first few records from the main record set (if present):
if record_set_ids:
    main_record_set_id = record_set_ids[0]
    print(f"Sample records from {main_record_set_id}:")
    for i, rec in enumerate(dataset.records(record_set=main_record_set_id)):
        print(rec)
        if i >= 2:
            break

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. All record sets and field `@id`s are referenced by their Croissant `@id`.

We extract all available record sets into pandas DataFrames.

In [ ]:
# Extract data from each record set

dataframes = {}
for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    df = pd.DataFrame(records)
    dataframes[record_set_id] = df
    print(f"Loaded {len(df)} records for record set {record_set_id}.")

# Show columns from the first record set as example
main_record_set_id = record_set_ids[0] if record_set_ids else None
if main_record_set_id is not None and not dataframes[main_record_set_id].empty:
    print(f"\nFirst record set ({main_record_set_id}) columns:")
    print(dataframes[main_record_set_id].columns.tolist())
    dataframes[main_record_set_id].head()

## 4. Exploratory Data Analysis (EDA)

Apply common data processing steps, such as filtering records based on specific criteria, normalizing numeric fields, and categorizing data. This can include removing outliers, transforming value distributions, or grouping records. We'll use field and record set `@id`s identified in previous steps.

Below, we select the PHQ-9 total score as a numeric field for demonstration (assuming its field or column is named 'phq9_total' or similar).

In [ ]:
# Choose appropriate field @id based on dataset description and printed columns

df = dataframes[main_record_set_id]
numeric_field = None
# Try to auto-detect commonly used mental health score fields
for col in df.columns:
    if 'phq9' in col.lower() and 'total' in col.lower():
        numeric_field = col
        break
if numeric_field is None:
    for col in df.columns:
        if 'gad7' in col.lower() and 'total' in col.lower():
            numeric_field = col
            break

if numeric_field is None:
    numeric_field = df.select_dtypes(include='number').columns[0] if len(df.select_dtypes(include='number').columns) else df.columns[0]

print(f"Numeric field for analysis: {numeric_field}")

# Set a sample threshold for filtering
threshold = 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}: {len(filtered_df)} rows")
display(filtered_df.head())

# Normalization
filtered_df[f"{numeric_field}_normalized"] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std(ddof=0)
print(f"Normalized {numeric_field} for filtered records:")
display(filtered_df[[numeric_field, f"{numeric_field}_normalized"]].head())

# Try grouping by a demographic field, e.g., 'gender' or 'sex' or similar
import numpy as np
group_field = None
for c in df.columns:
    if c.lower() in ['gender', 'sex', 'marital_status', 'village', 'age_group', 'level_of_education']:
        group_field = c
        break

if group_field is not None:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].agg(['mean', 'count', np.median])
    print(f"Grouped data by {group_field} (mean/count/median):")
    display(grouped_df.head())
else:
    print("No categorical demographic field found for grouping.")

## 5. Visualization
Visualize distributions and relationships in the dataset. This includes histogram of mental health scores and a box plot by a demographic grouping if present.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Histogram of the main mental health score
plt.figure(figsize=(7,4))
sns.histplot(df[numeric_field].dropna(), bins=15, kde=True, color='salmon')
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# Boxplot by group, if group_field detected
if group_field is not None:
    plt.figure(figsize=(9,5))
    sns.boxplot(x=group_field, y=numeric_field, data=df, showmeans=True, palette='pastel')
    plt.title(f"{numeric_field} distribution by {group_field}")
    plt.xticks(rotation=45)
    plt.show()
else:
    print('No group field for boxplot.')

## 6. Conclusion

This notebook demonstrated the workflow for loading a Croissant-described dataset, extracting its structure and data, and conducting basic analysis and visualization using Python.

**Key observations:**
- The dataset provides individual-level mental health and demographic data from Kilifi County, Kenya.
- Numeric mental health assessment fields (e.g., PHQ-9 and GAD-7 scores) can be filtered and analyzed by demographic attributes.
- Visual exploration reveals symptom burden distributions and possible group differences (e.g., by gender or education level).

Researchers are encouraged to further explore other fields, examine missing data, and combine with contextual information for advanced analyses.